# Qwen layer-by-image-region analysis

This is the cheap exploratory analysis for the question:

> Do different hidden-state depths read different parts of the image?

It reuses existing outputs and performs no model inference.

## Required files in the same folder

- `pope_confidence_category_matched_RESULTS_20260718_062034.zip`
- `qwen_semantic_causal_hallucination_diagnosis_RESULTS.zip`

## What it produces

1. Qwen 7B layer-by-patch sensitivity matrices.
2. The most influential patch at each hidden-state depth.
3. Top-patch stability between neighboring layers.
4. Heatmap concentration or diffuseness across depth.
5. Grounded-example target overlap across depth.
6. Qwen 7B and 3B semantic target-concept signal across depth.
7. An exploratory report and output ZIP.

This is not yet activation patching. It shows where layerwise sensitivity and semantic readout change. A stronger causal experiment should only be run if these plots show a clear pattern.


In [ ]:
# Cell 1: imports and paths

import json
import math
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score

ROOT = Path.cwd()
OUT = ROOT / "qwen_layer_by_image_region_analysis"
EXTRACT = OUT / "extracted"
FIGURES = OUT / "figures"

for directory in [OUT, EXTRACT, FIGURES]:
    directory.mkdir(parents=True, exist_ok=True)

ZIP_7B = ROOT / "pope_confidence_category_matched_RESULTS_20260718_062034.zip"
ZIP_SEMANTIC = ROOT / "qwen_semantic_causal_hallucination_diagnosis_RESULTS.zip"

for path in [ZIP_7B, ZIP_SEMANTIC]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path.name}. Put it beside this notebook."
        )

def extract_once(source: Path, destination: Path) -> None:
    marker = destination / ".complete"
    if marker.exists():
        return
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source) as archive:
        archive.extractall(destination)
    marker.write_text(source.name)

extract_once(ZIP_7B, EXTRACT / "qwen7b")
extract_once(ZIP_SEMANTIC, EXTRACT / "semantic")

print("Ready:", OUT)


In [ ]:
# Cell 2: load and normalize the saved all-layer outputs

layer_path = EXTRACT / "qwen7b" / "outputs" / "grid_patch_layer_distances.csv"
selected_path = EXTRACT / "qwen7b" / "outputs" / "selected_confidence_category_matched.csv"

semantic_candidates = [
    EXTRACT / "semantic" / "qwen7b_semantic_probe.csv",
    EXTRACT / "semantic" / "qwen3b_semantic_probe.csv",
]

if not layer_path.exists():
    raise FileNotFoundError(layer_path)
if not selected_path.exists():
    raise FileNotFoundError(selected_path)

distances = pd.read_csv(layer_path)
selected = pd.read_csv(selected_path)

required_columns = {
    "selected_index",
    "layer",
    "patch_index",
    "grid_x",
    "grid_y",
    "relative_l2",
    "clean_group",
    "patch_has_target",
}
missing = required_columns - set(distances.columns)
if missing:
    raise RuntimeError(f"Missing columns: {sorted(missing)}")

for column in ["selected_index", "layer", "patch_index", "grid_x", "grid_y"]:
    distances[column] = pd.to_numeric(distances[column], errors="raise").astype(int)

distances["relative_l2"] = pd.to_numeric(
    distances["relative_l2"], errors="coerce"
).fillna(0.0)

if "patch_has_target" in distances.columns:
    distances["patch_has_target"] = pd.to_numeric(
        distances["patch_has_target"], errors="coerce"
    ).fillna(0).astype(int)

semantic_frames = []
for path in semantic_candidates:
    if path.exists():
        frame = pd.read_csv(path)
        frame["model_name"] = path.stem.split("_semantic_probe")[0]
        semantic_frames.append(frame)

semantic = (
    pd.concat(semantic_frames, ignore_index=True)
    if semantic_frames
    else pd.DataFrame()
)

print("7B all-layer patch rows:", len(distances))
print("7B examples:", distances.selected_index.nunique())
print("7B hidden-state indices:", distances.layer.nunique())
print("Semantic rows:", len(semantic))


In [ ]:
# Cell 3: aggregate layerwise spatial metrics

def normalized_entropy(values):
    values = np.asarray(values, dtype=float)
    values = values - np.nanmin(values)
    values = values + 1e-12
    probabilities = values / values.sum()
    entropy = -np.sum(probabilities * np.log(probabilities))
    return float(entropy / np.log(len(probabilities)))

per_example_rows = []

for (selected_index, layer), group in distances.groupby(
    ["selected_index", "layer"], sort=True
):
    group = group.sort_values("patch_index")
    scores = group.relative_l2.to_numpy(float)
    ranking = np.argsort(-scores, kind="stable")
    top1 = int(ranking[0])
    top3 = ranking[:3].astype(int)

    target = group.patch_has_target.to_numpy(int)
    patch_ap = np.nan
    top1_hit = np.nan
    top3_hit = np.nan

    if target.sum() > 0 and target.sum() < len(target):
        patch_ap = float(average_precision_score(target, scores))
        top1_hit = float(target[top1] > 0)
        top3_hit = float(target[top3].max() > 0)

    per_example_rows.append({
        "selected_index": int(selected_index),
        "layer": int(layer),
        "clean_group": str(group.clean_group.iloc[0]),
        "top1_patch": top1,
        "top1_grid_x": int(top1 % 6),
        "top1_grid_y": int(top1 // 6),
        "top3_json": json.dumps(top3.tolist()),
        "normalized_entropy": normalized_entropy(scores),
        "top1_mass": float(
            max(scores[top1], 0.0)
            / (np.clip(scores, 0.0, None).sum() + 1e-12)
        ),
        "patch_ap": patch_ap,
        "top1_target_hit": top1_hit,
        "top3_target_hit": top3_hit,
    })

per_example = pd.DataFrame(per_example_rows)

# Adjacent-layer top-patch stability
stability_rows = []
for selected_index, group in per_example.groupby("selected_index"):
    group = group.sort_values("layer")
    previous_patch = None
    for _, row in group.iterrows():
        stable = np.nan if previous_patch is None else float(
            int(row.top1_patch) == int(previous_patch)
        )
        stability_rows.append({
            "selected_index": int(selected_index),
            "layer": int(row.layer),
            "adjacent_top1_stability": stable,
        })
        previous_patch = int(row.top1_patch)

stability = pd.DataFrame(stability_rows)
per_example = per_example.merge(
    stability,
    on=["selected_index", "layer"],
    how="left",
)

layer_summary = (
    per_example.groupby("layer")
    .agg(
        n=("selected_index", "nunique"),
        mean_entropy=("normalized_entropy", "mean"),
        mean_top1_mass=("top1_mass", "mean"),
        adjacent_top1_stability=("adjacent_top1_stability", "mean"),
        grounded_patch_ap=("patch_ap", "mean"),
        grounded_top1_hit=("top1_target_hit", "mean"),
        grounded_top3_hit=("top3_target_hit", "mean"),
    )
    .reset_index()
)

per_example.to_csv(OUT / "per_example_layer_metrics.csv", index=False)
layer_summary.to_csv(OUT / "aggregate_layer_metrics.csv", index=False)

print(layer_summary.head().to_string(index=False))


In [ ]:
# Cell 4: figures and representative layer-by-patch matrices

# Aggregate curves
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    layer_summary.layer,
    layer_summary.grounded_patch_ap,
    marker="o",
    markersize=3,
    label="Grounded patch AP",
)
ax.plot(
    layer_summary.layer,
    layer_summary.grounded_top1_hit,
    marker="o",
    markersize=3,
    label="Grounded top-1 hit",
)
ax.plot(
    layer_summary.layer,
    layer_summary.grounded_top3_hit,
    marker="o",
    markersize=3,
    label="Grounded top-3 hit",
)
ax.set_xlabel("Hidden-state index")
ax.set_ylabel("Localization performance")
ax.set_ylim(0, 1)
ax.set_title("Where target localization emerges across Qwen 7B depth")
ax.legend()
fig.tight_layout()
fig.savefig(
    FIGURES / "localization_across_depth.png",
    dpi=220,
    bbox_inches="tight",
)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    layer_summary.layer,
    layer_summary.mean_entropy,
    marker="o",
    markersize=3,
    label="Normalized heatmap entropy",
)
ax.plot(
    layer_summary.layer,
    layer_summary.adjacent_top1_stability,
    marker="o",
    markersize=3,
    label="Adjacent-layer top-patch stability",
)
ax.set_xlabel("Hidden-state index")
ax.set_ylabel("Value")
ax.set_ylim(0, 1)
ax.set_title("Spatial concentration and patch stability across depth")
ax.legend()
fig.tight_layout()
fig.savefig(
    FIGURES / "spatial_dynamics_across_depth.png",
    dpi=220,
    bbox_inches="tight",
)
plt.close(fig)

# Select representative examples:
# known airport case when present, plus high-confidence examples from each group.
metadata = selected[
    [
        "selected_index",
        "row_idx",
        "question",
        "target_category",
        "clean_group",
        "clean_p_yes",
    ]
].drop_duplicates("selected_index")

example_scores = (
    per_example.groupby("selected_index")
    .agg(
        mean_entropy=("normalized_entropy", "mean"),
        mean_top1_mass=("top1_mass", "mean"),
        mean_stability=("adjacent_top1_stability", "mean"),
    )
    .reset_index()
    .merge(metadata, on="selected_index", how="left")
)

representative_ids = []

airport = example_scores[
    example_scores.row_idx.astype(int) == 5931
]
if len(airport):
    representative_ids.append(int(airport.iloc[0].selected_index))

for group_name in ["grounded_yes", "hallucinated_yes"]:
    candidates = example_scores[
        example_scores.clean_group.astype(str) == group_name
    ].sort_values(
        ["mean_top1_mass", "mean_stability"],
        ascending=[False, False],
    )
    for value in candidates.selected_index.head(5):
        value = int(value)
        if value not in representative_ids:
            representative_ids.append(value)

representative_ids = representative_ids[:10]

for selected_index in representative_ids:
    frame = distances[
        distances.selected_index.astype(int) == selected_index
    ].copy()

    matrix = (
        frame.pivot(
            index="layer",
            columns="patch_index",
            values="relative_l2",
        )
        .sort_index()
        .sort_index(axis=1)
        .to_numpy(float)
    )

    # Normalize each layer independently to emphasize spatial changes.
    row_min = matrix.min(axis=1, keepdims=True)
    row_max = matrix.max(axis=1, keepdims=True)
    normalized = (matrix - row_min) / (row_max - row_min + 1e-12)

    meta = metadata[
        metadata.selected_index.astype(int) == selected_index
    ].iloc[0]

    fig, ax = plt.subplots(figsize=(12, 6))
    image = ax.imshow(
        normalized,
        aspect="auto",
        interpolation="nearest",
    )
    ax.set_xlabel("Image patch index (6 x 6 grid)")
    ax.set_ylabel("Hidden-state index")
    ax.set_title(
        f"{meta.clean_group}: {meta.question}\n"
        f"target={meta.target_category}, row={int(meta.row_idx)}"
    )
    fig.colorbar(image, ax=ax, label="Within-layer normalized sensitivity")
    fig.tight_layout()
    fig.savefig(
        FIGURES / f"layer_by_patch_selected_{selected_index}.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.close(fig)

print("Representative examples:", representative_ids)


In [ ]:
# Cell 5: semantic signal across depth for Qwen 7B and 3B

semantic_summary = pd.DataFrame()

if not semantic.empty:
    semantic["layer"] = pd.to_numeric(
        semantic["layer"], errors="raise"
    ).astype(int)
    semantic["target_margin"] = pd.to_numeric(
        semantic["target_margin"], errors="coerce"
    )

    rows = []
    for (model_name, layer), group in semantic.groupby(
        ["model_name", "layer"], sort=True
    ):
        hallucinated = group[
            group.clean_group.astype(str) == "hallucinated_yes"
        ]
        grounded = group[
            group.clean_group.astype(str) == "grounded_yes"
        ]
        rows.append({
            "model_name": model_name,
            "layer": int(layer),
            "layer_fraction": float(group.layer_fraction.mean()),
            "mean_margin_all": float(group.target_margin.mean()),
            "positive_rate_all": float(
                (group.target_margin > 0).mean()
            ),
            "mean_margin_grounded": float(
                grounded.target_margin.mean()
            ),
            "mean_margin_hallucinated": float(
                hallucinated.target_margin.mean()
            ),
            "positive_rate_hallucinated": float(
                (hallucinated.target_margin > 0).mean()
            ),
        })

    semantic_summary = pd.DataFrame(rows)
    semantic_summary.to_csv(
        OUT / "semantic_signal_across_depth.csv",
        index=False,
    )

    for metric, label in [
        ("mean_margin_hallucinated", "Mean target-concept margin"),
        (
            "positive_rate_hallucinated",
            "Fraction causal patch > random patch",
        ),
    ]:
        fig, ax = plt.subplots(figsize=(9, 5))
        for model_name, group in semantic_summary.groupby("model_name"):
            group = group.sort_values("layer_fraction")
            ax.plot(
                group.layer_fraction,
                group[metric],
                marker="o",
                markersize=3,
                label=model_name,
            )
        ax.axhline(0 if "margin" in metric else 0.5, linestyle="--")
        ax.set_xlabel("Relative model depth")
        ax.set_ylabel(label)
        ax.set_title(
            "Queried-object semantic signal at the causal patch "
            "across depth"
        )
        ax.legend()
        fig.tight_layout()
        fig.savefig(
            FIGURES / f"{metric}_across_depth.png",
            dpi=220,
            bbox_inches="tight",
        )
        plt.close(fig)

print("Semantic layer analysis complete:", len(semantic_summary))


In [ ]:
# Cell 6: exploratory report and export

def best_layer(frame, column, maximize=True):
    clean = frame[["layer", column]].dropna()
    if clean.empty:
        return None
    index = clean[column].idxmax() if maximize else clean[column].idxmin()
    row = clean.loc[index]
    return {
        "layer": int(row.layer),
        "value": float(row[column]),
    }

report = {
    "analysis_type": (
        "Exploratory layer-by-image-region sensitivity analysis. "
        "Not activation patching."
    ),
    "qwen7b_examples": int(per_example.selected_index.nunique()),
    "qwen7b_hidden_state_indices": int(per_example.layer.nunique()),
    "best_grounded_patch_ap_layer": best_layer(
        layer_summary,
        "grounded_patch_ap",
    ),
    "most_spatially_concentrated_layer": best_layer(
        layer_summary,
        "mean_entropy",
        maximize=False,
    ),
    "most_adjacently_stable_layer": best_layer(
        layer_summary,
        "adjacent_top1_stability",
    ),
    "interpretation_rule": (
        "Proceed to activation patching only if the plots show a "
        "coherent change in spatial sensitivity or semantic signal "
        "across depth that is consistent across examples."
    ),
}

if not semantic_summary.empty:
    report["semantic"] = {}
    for model_name, group in semantic_summary.groupby("model_name"):
        report["semantic"][model_name] = {
            "best_hallucination_margin_depth": best_layer(
                group,
                "mean_margin_hallucinated",
            ),
            "best_hallucination_positive_rate_depth": best_layer(
                group,
                "positive_rate_hallucinated",
            ),
        }

with open(OUT / "layer_by_region_summary.json", "w") as file:
    json.dump(report, file, indent=2)

readme = f"""# Layer-by-image-region exploratory analysis

This analysis reuses saved Qwen outputs and performs no model inference.

## Main question

Do different hidden-state depths show sensitivity to different image regions?

## Outputs

- `aggregate_layer_metrics.csv`
- `per_example_layer_metrics.csv`
- `semantic_signal_across_depth.csv`
- `figures/localization_across_depth.png`
- `figures/spatial_dynamics_across_depth.png`
- representative `layer_by_patch_*.png` matrices
- semantic readout curves for Qwen 7B and 3B

## Important limitation

This analysis measures layerwise sensitivity and semantic readout. It does not
prove that a layer causally reads a patch. True activation patching would
require swapping visual-token activations between original and counterfactual
runs at each layer and measuring answer-logit recovery.

## Decision rule

Only run the more expensive activation-patching experiment when these plots
show a clear, reproducible transition across depth.
"""
(OUT / "README.md").write_text(readme)

archive = shutil.make_archive(
    str(ROOT / "qwen_layer_by_image_region_analysis_RESULTS"),
    "zip",
    root_dir=OUT,
)

print(json.dumps(report, indent=2))
print("\nOutput ZIP:", archive)
